<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_29_AI_EVALUATION_%26_MEASUREMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Day 29 — AI Evaluation & Measurement 🤖📊

## Focus Area
**AI Evaluation and Measurement using LLM-as-a-Judge**

An AI system ko improve karne ke liye sirf answers generate karna enough nahi hai.
Humein objectively measure karna bhi zaroori hai ki answers kitne reliable hain.

In this project, I built an automated evaluation framework for a knowledge
assistant using a local LLM as an evaluator.

---

## 🎯 Evaluation Dimensions

### 1. Groundedness
Checks whether the assistant's answer is supported by the retrieved context
and does not contain unsupported or fabricated information.

**Scale:** 1–5

### 2. Correctness
Checks whether the assistant's answer matches the manually prepared
ground-truth answer.

**Scale:** 1–5

### 3. Completeness
Checks whether the assistant fully answers the question without missing
important information.

**Scale:** 1–5

---

## 🔄 Evaluation Workflow

```text
Question
   ↓
Knowledge Assistant
   ↓
Retrieved Context + Generated Answer
   ↓
LLM-as-Judge
   ↓
┌─────────────────┬─────────────────┬─────────────────┐
│  Groundedness   │   Correctness   │   Completeness  │
│      1–5        │       1–5       │       1–5       │
└─────────────────┴─────────────────┴─────────────────┘
   ↓
Aggregate Scores
   ↓
Regression Testing
   ↓
PASS / FAIL

In [2]:
# ============================================================
# DAY 29: LLM-AS-JUDGE EVALUATION
# Google Colab - WITHOUT OPENAI API
# ============================================================

# Install required libraries
!pip -q install transformers accelerate torch sentencepiece


# ============================================================
# 1. IMPORTS
# ============================================================

import json
import re
import statistics
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# 2. LOAD LOCAL LLM
# ============================================================

# Small instruction-following model
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading LLM...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("✅ LLM loaded")


# ============================================================
# 3. FUNCTION TO CALL LOCAL LLM
# ============================================================

def ask_llm(prompt, max_tokens=300):

    messages = [
        {
            "role": "system",
            "content": "You are a strict and objective AI evaluator."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=False
        )

    generated_tokens = outputs[
        0
    ][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# ============================================================
# 4. EVALUATION CRITERIA
# ============================================================

criteria = """

GROUNDedness:
Check whether the assistant answer is supported ONLY by
the provided context.

1 = Mostly unsupported / hallucinated
2 = Many unsupported claims
3 = Partially supported
4 = Mostly supported
5 = Completely supported by context

CORRECTNESS:
Check whether the answer agrees with the ground truth.

1 = Completely incorrect
2 = Mostly incorrect
3 = Partially correct
4 = Mostly correct
5 = Fully correct

COMPLETENESS:
Check whether the answer addresses all important parts
of the question.

1 = Almost nothing answered
2 = Major information missing
3 = Partially complete
4 = Mostly complete
5 = Fully complete
"""


# ============================================================
# 5. LLM-AS-JUDGE FUNCTION
# ============================================================

def llm_judge(
    question,
    context,
    answer,
    ground_truth
):

    prompt = f"""
You are evaluating an AI knowledge assistant.

Evaluate the assistant answer on exactly THREE dimensions.

{criteria}

QUESTION:
{question}

RETRIEVED CONTEXT:
{context}

ASSISTANT ANSWER:
{answer}

GROUND TRUTH:
{ground_truth}

Return ONLY a JSON object.

Required format:

{{
    "groundedness": 1,
    "correctness": 1,
    "completeness": 1
}}

Each value MUST be an integer between 1 and 5.
"""

    response = ask_llm(
        prompt,
        max_tokens=150
    )

    # --------------------------------------------------------
    # Extract JSON from LLM response
    # --------------------------------------------------------

    match = re.search(
        r'\{.*?\}',
        response,
        re.DOTALL
    )

    if not match:

        print("⚠️ Judge returned:")
        print(response)

        return {
            "groundedness": 1,
            "correctness": 1,
            "completeness": 1
        }

    json_text = match.group()

    try:

        scores = json.loads(json_text)

    except:

        print("⚠️ Could not parse JSON:")
        print(response)

        return {
            "groundedness": 1,
            "correctness": 1,
            "completeness": 1
        }

    # Validate scores

    for dimension in [
        "groundedness",
        "correctness",
        "completeness"
    ]:

        if dimension not in scores:
            scores[dimension] = 1

        scores[dimension] = max(
            1,
            min(
                5,
                int(scores[dimension])
            )
        )

    return scores


# ============================================================
# 6. DEMO KNOWLEDGE BASE
# ============================================================

knowledge_base = {

    "python":
    """
    Python is a high-level interpreted programming language.
    It was created by Guido van Rossum and first released in 1991.
    """,

    "oop":
    """
    Object-oriented programming organizes software around objects
    that contain data and behavior.
    Four commonly discussed principles are encapsulation,
    abstraction, inheritance, and polymorphism.
    """,

    "dbms":
    """
    A DBMS is software used to create, store, organize,
    retrieve, and manage data in databases.
    """,

    "sql":
    """
    SQL stands for Structured Query Language.
    SQL is commonly used to create, retrieve, update,
    and delete data in relational databases.
    """,

    "ml":
    """
    Machine learning is a branch of artificial intelligence
    where systems learn patterns from data to make predictions
    or decisions without being explicitly programmed for
    every individual task.
    """,

    "supervised":
    """
    Supervised learning uses labelled training data.
    Classification and regression are common supervised
    learning tasks.
    """,

    "unsupervised":
    """
    Unsupervised learning works with data that does not have
    labelled target values.
    Clustering and dimensionality reduction are common examples.
    """,

    "rag":
    """
    Retrieval-Augmented Generation, or RAG, combines information
    retrieval with language generation.
    Relevant information is retrieved and provided as context
    to a language model before generating an answer.
    """,

    "embedding":
    """
    Embeddings are numerical vector representations of text
    or other data.
    Similarity between embeddings can be used to find
    semantically related information.
    """,

    "api":
    """
    APIs allow different software systems to communicate.
    REST APIs commonly use HTTP methods such as GET, POST,
    PUT, PATCH, and DELETE.
    """
}


# ============================================================
# 7. SIMPLE RETRIEVER
# ============================================================

def retrieve_context(question):

    question = question.lower()

    for key in knowledge_base:

        if key in question:

            return knowledge_base[key]

    # keyword matching

    words = question.split()

    best_doc = ""

    best_score = 0

    for key, document in knowledge_base.items():

        score = sum(
            word in document.lower()
            for word in words
            if len(word) > 3
        )

        if score > best_score:

            best_score = score
            best_doc = document

    return best_doc


# ============================================================
# 8. AI ASSISTANT
# ============================================================

def ask_assistant(question):

    context = retrieve_context(question)

    prompt = f"""
You are a knowledge assistant.

Answer the question using ONLY the provided context.

Do not invent information.

If the context does not contain enough information,
say that the information is not available.

CONTEXT:
{context}

QUESTION:
{question}

Answer:
"""

    answer = ask_llm(
        prompt,
        max_tokens=150
    )

    return {
        "question": question,
        "context": context,
        "answer": answer
    }


# ============================================================
# 9. 20 QUESTION EVALUATION DATASET
# ============================================================

evaluation_dataset = [

    {
        "id": 1,
        "question": "What is Python?",
        "ground_truth":
        "Python is a high-level interpreted programming language."
    },

    {
        "id": 2,
        "question": "Who created Python?",
        "ground_truth":
        "Python was created by Guido van Rossum."
    },

    {
        "id": 3,
        "question": "When was Python first released?",
        "ground_truth":
        "Python was first released in 1991."
    },

    {
        "id": 4,
        "question":
        "What are the four principles of object-oriented programming?",
        "ground_truth":
        "The four commonly discussed principles are encapsulation, abstraction, inheritance, and polymorphism."
    },

    {
        "id": 5,
        "question":
        "What is object-oriented programming?",
        "ground_truth":
        "Object-oriented programming organizes software around objects that contain data and behavior."
    },

    {
        "id": 6,
        "question": "What is a DBMS?",
        "ground_truth":
        "A DBMS is software used to create, store, organize, retrieve, and manage data in databases."
    },

    {
        "id": 7,
        "question": "What does SQL stand for?",
        "ground_truth":
        "SQL stands for Structured Query Language."
    },

    {
        "id": 8,
        "question": "What is SQL used for?",
        "ground_truth":
        "SQL is used to create, retrieve, update, and delete data in relational databases."
    },

    {
        "id": 9,
        "question": "What is machine learning?",
        "ground_truth":
        "Machine learning is a branch of artificial intelligence where systems learn patterns from data to make predictions or decisions."
    },

    {
        "id": 10,
        "question": "What is supervised learning?",
        "ground_truth":
        "Supervised learning uses labelled training data."
    },

    {
        "id": 11,
        "question":
        "What are common supervised learning tasks?",
        "ground_truth":
        "Classification and regression are common supervised learning tasks."
    },

    {
        "id": 12,
        "question": "What is unsupervised learning?",
        "ground_truth":
        "Unsupervised learning works with data that does not have labelled target values."
    },

    {
        "id": 13,
        "question":
        "Give examples of unsupervised learning.",
        "ground_truth":
        "Clustering and dimensionality reduction are common examples."
    },

    {
        "id": 14,
        "question": "What is RAG?",
        "ground_truth":
        "RAG combines information retrieval with language generation."
    },

    {
        "id": 15,
        "question":
        "What does RAG provide to a language model?",
        "ground_truth":
        "RAG provides retrieved relevant information as context to the language model."
    },

    {
        "id": 16,
        "question": "What are embeddings?",
        "ground_truth":
        "Embeddings are numerical vector representations of text or other data."
    },

    {
        "id": 17,
        "question":
        "How can embeddings be used?",
        "ground_truth":
        "Similarity between embeddings can be used to find semantically related information."
    },

    {
        "id": 18,
        "question": "What is an API?",
        "ground_truth":
        "An API allows different software systems to communicate with each other."
    },

    {
        "id": 19,
        "question":
        "Which HTTP methods are commonly used by REST APIs?",
        "ground_truth":
        "GET, POST, PUT, PATCH, and DELETE are commonly used HTTP methods."
    },

    {
        "id": 20,
        "question":
        "What is the purpose of a DBMS?",
        "ground_truth":
        "A DBMS is used to create, store, organize, retrieve, and manage data in databases."
    }
]


# ============================================================
# 10. RUN ALL 20 TESTS
# ============================================================

results = {}

print("\n")
print("=" * 70)
print("RUNNING 20 QUESTION EVALUATION")
print("=" * 70)


for item in evaluation_dataset:

    print(
        f"\nEvaluating {item['id']}/20..."
    )

    assistant_result = ask_assistant(
        item["question"]
    )

    scores = llm_judge(
        question=item["question"],
        context=assistant_result["context"],
        answer=assistant_result["answer"],
        ground_truth=item["ground_truth"]
    )

    results[item["id"]] = {

        "question":
        item["question"],

        "ground_truth":
        item["ground_truth"],

        "context":
        assistant_result["context"],

        "answer":
        assistant_result["answer"],

        "scores":
        scores
    }


# ============================================================
# 11. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("INDIVIDUAL EVALUATION RESULTS")
print("=" * 70)


for question_id, result in results.items():

    scores = result["scores"]

    print(
        f"\nQ{question_id}: "
        f"{result['question']}"
    )

    print(
        "Answer:",
        result["answer"]
    )

    print(
        "Groundedness:",
        scores["groundedness"],
        "/5"
    )

    print(
        "Correctness:",
        scores["correctness"],
        "/5"
    )

    print(
        "Completeness:",
        scores["completeness"],
        "/5"
    )


# ============================================================
# 12. AGGREGATE SCORES
# ============================================================

dimensions = [
    "groundedness",
    "correctness",
    "completeness"
]

aggregate_scores = {}


for dimension in dimensions:

    scores = [
        result["scores"][dimension]
        for result in results.values()
    ]

    aggregate_scores[dimension] = round(
        statistics.mean(scores),
        2
    )


print("\n")
print("=" * 70)
print("AGGREGATE SCORE REPORT")
print("=" * 70)


for dimension, score in aggregate_scores.items():

    print(
        f"{dimension.capitalize():15} "
        f"{score}/5"
    )


# ============================================================
# 13. LOWEST DIMENSION
# ============================================================

lowest_dimension = min(
    aggregate_scores,
    key=aggregate_scores.get
)

highest_dimension = max(
    aggregate_scores,
    key=aggregate_scores.get
)

gap = round(
    aggregate_scores[highest_dimension]
    -
    aggregate_scores[lowest_dimension],
    2
)


print("\nLowest Dimension:")
print(
    lowest_dimension,
    "=",
    aggregate_scores[lowest_dimension],
    "/5"
)

print(
    "Difference from highest:",
    gap,
    "points"
)


# ============================================================
# 14. REGRESSION TEST
# ============================================================

# Current scores ko baseline save kar rahe hain
baseline = aggregate_scores.copy()


def regression_test_runner(
    baseline,
    current_scores,
    threshold=0.3
):

    print("\n")
    print("=" * 70)
    print("REGRESSION TEST")
    print("=" * 70)

    overall_pass = True

    for dimension in dimensions:

        old_score = baseline[dimension]

        new_score = current_scores[dimension]

        drop = round(
            old_score - new_score,
            2
        )

        if drop > threshold:

            print(
                f"❌ FAIL | "
                f"{dimension} | "
                f"Score dropped by {drop}"
            )

            overall_pass = False

        else:

            print(
                f"✅ PASS | "
                f"{dimension} | "
                f"Score drop = {drop}"
            )

    print("-" * 70)

    if overall_pass:

        print(
            "FINAL RESULT: ✅ PASS"
        )

    else:

        print(
            "FINAL RESULT: ❌ FAIL"
        )

    return overall_pass


# Run regression test

regression_test_runner(
    baseline,
    aggregate_scores,
    threshold=0.3
)


# ============================================================
# 15. SAVE RESULTS
# ============================================================

with open(
    "evaluation_results.json",
    "w"
) as file:

    json.dump(
        results,
        file,
        indent=4
    )


with open(
    "evaluation_baseline.json",
    "w"
) as file:

    json.dump(
        baseline,
        file,
        indent=4
    )


print("\n")
print("✅ evaluation_results.json saved")
print("✅ evaluation_baseline.json saved")

print("\n")
print("=" * 70)
print("DAY 29 COMPLETED")
print("=" * 70)

Loading LLM...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ LLM loaded


RUNNING 20 QUESTION EVALUATION

Evaluating 1/20...

Evaluating 2/20...

Evaluating 3/20...

Evaluating 4/20...

Evaluating 5/20...

Evaluating 6/20...

Evaluating 7/20...

Evaluating 8/20...

Evaluating 9/20...

Evaluating 10/20...

Evaluating 11/20...

Evaluating 12/20...

Evaluating 13/20...

Evaluating 14/20...

Evaluating 15/20...

Evaluating 16/20...

Evaluating 17/20...

Evaluating 18/20...

Evaluating 19/20...

Evaluating 20/20...


INDIVIDUAL EVALUATION RESULTS

Q1: What is Python?
Answer: Python is a high-level, interpreted programming language.
Groundedness: 1 /5
Correctness: 1 /5
Completeness: 1 /5

Q2: Who created Python?
Answer: Python was created by Guido van Rossum.
Groundedness: 1 /5
Correctness: 1 /5
Completeness: 1 /5

Q3: When was Python first released?
Answer: 1991
Groundedness: 3 /5
Correctness: 5 /5
Completeness: 5 /5

Q4: What are the four principles of object-oriented programming?
Answer: encapsulation, abstraction, inheritance, and polymorphism.
